In [ ]:
# Compare: time the viewer before and after a change, in the same Colab session.
# Colab's speed drifts from one session to the next, so the old and the new viewer are
# built here side by side and play the same walk in turns, on each scene. Needs a T4 GPU runtime.
import csv, os, shutil, statistics, struct, subprocess, sys

REPO_URL = "https://github.com/ShreyasxAnand/pointwalk"
BRANCH = "dev"
BEFORE = "6929088"   # the commit to compare against: the viewer before the change
ROUNDS = 2           # each version plays the walk this many times, taking turns
SCENES = ["synthetic", "stanford"]   # which scenes to time on
FLAGS = ["-O2", "-arch=sm_75", "-fmad=false"]  # -fmad=false: CPU and GPU match bit for bit

# The Stanford scene: the same request as experiments/realplace/stanford.ipynb.
SURVEY = "CA_SantaClaraCounty_2020"
PLACE = dict(lat=37.4275, lon=-122.1700)
SIZE_M = 1000
DEPTH = 11           # 11 = 23 million dots

repo = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
build = os.path.join(repo, "build_compare")
exp = os.path.join(repo, "experiments", "realplace")


def sh(*cmd, cwd=None, quiet=False):
    """Run a command, print its output, stop the cell if it fails."""
    print("$ " + " ".join(cmd), flush=True)
    r = subprocess.run(cmd, cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if not quiet or r.returncode != 0:
        print(r.stdout, flush=True)
    if r.returncode != 0:
        raise SystemExit(f"FAILED (exit {r.returncode}): {' '.join(cmd)}")
    return r.stdout


# 1. Get the code.
if not os.path.isdir(os.path.join(repo, ".git")):
    sh("git", "clone", "--branch", BRANCH, REPO_URL, repo)
else:
    sh("git", "fetch", "origin", BRANCH, cwd=repo)
    sh("git", "checkout", BRANCH, cwd=repo)
    sh("git", "pull", "--ff-only", "origin", BRANCH, cwd=repo)
after = sh("git", "log", "-1", "--format=%h", cwd=repo, quiet=True).strip()
sh("git", "log", "-1", "--format=%h %ad %s", "--date=iso", cwd=repo)   # the "after" version
sh("nvidia-smi", "--query-gpu=name", "--format=csv,noheader")

# 2. Put a copy of the old source next to the new one.
shutil.rmtree(build, ignore_errors=True)
os.makedirs(build)
old_source = os.path.join(build, "before_source")
sh("git", "worktree", "prune", cwd=repo)
sh("git", "worktree", "add", "--detach", old_source, BEFORE, cwd=repo, quiet=True)

# 3. Compile. The new project program runs once: it checks GPU against CPU and saves the synthetic scene.
folders = {"before": os.path.join(build, "before"), "after": os.path.join(build, "after")}
for folder in folders.values():
    os.makedirs(folder)
sh("nvcc", *FLAGS, os.path.join(old_source, "src", "viewer.cu"), "-o", os.path.join(folders["before"], "viewer"))
sh("nvcc", *FLAGS, os.path.join(repo, "src", "viewer.cu"), "-o", os.path.join(folders["after"], "viewer"))
sh("nvcc", *FLAGS, os.path.join(repo, "src", "project.cu"), "-o", os.path.join(folders["after"], "project"))
out = sh("./project", cwd=folders["after"], quiet=True)
print("\n".join(line for line in out.splitlines() if "differ" in line), "\n")

# 4. The scenes: (name, scene file, file with the starting viewpoint or None).
scenes = []
if "synthetic" in SCENES:
    scenes.append(("synthetic floor and sphere", os.path.join(folders["after"], "scene.bin"), None))
if "stanford" in SCENES:
    # Kept in build_real, where the Stanford notebook keeps it, so it is fetched only once.
    real = os.path.join(repo, "build_real")
    os.makedirs(real, exist_ok=True)
    scene = os.path.join(real, f"{SURVEY}_{PLACE['lat']}_{PLACE['lon']}_{SIZE_M}_{DEPTH}.bin")
    if not os.path.exists(scene + ".pose"):
        sh(sys.executable, "-m", "pip", "install", "-q", "laspy[lazrs]", quiet=True)
        sh(sys.executable, os.path.join(exp, "fetch_place.py"), "--survey", SURVEY,
           "--lat", str(PLACE["lat"]), "--lon", str(PLACE["lon"]),
           "--size", str(SIZE_M), "--depth", str(DEPTH), "--out", scene)
    else:
        print("using the scene already on disk:", scene)
    # The walk starts from the Stanford viewer's opening view: up high, looking down at the centre.
    scenes.append(("Stanford lidar", scene, scene + ".pose"))

# 5. On each scene, both viewers play the same walk in turns. Keep the middle time of each run.
COLUMNS = [("kernel_ms", "the two kernels"),
           ("gpu_picture_ms", "whole picture on GPU"),
           ("round_trip_ms", "handed to script")]
names = {"before": f"before {BEFORE}", "after": f"after  {after}"}
tables = []
for scene_number, (scene_name, scene, pose_file) in enumerate(scenes):
    with open(scene, "rb") as f:
        dots = struct.unpack("<i", f.read(12)[8:])[0]      # the dot count sits after the 8 letter tag
    results = {"before": [], "after": []}
    for round_number in range(1, ROUNDS + 1):
        for version in ["before", "after"]:
            out_dir = os.path.join(folders[version], f"scene{scene_number}_round{round_number}")
            os.makedirs(out_dir)
            command = [sys.executable, os.path.join(repo, "viewer", "replay.py"),
                       "--renderer", os.path.join(folders[version], "viewer"),
                       "--scene", scene, "--out", out_dir]
            if pose_file:
                command += ["--start-pose", pose_file]
            sh(*command, quiet=True)
            with open(os.path.join(out_dir, "walk_timing.csv")) as f:
                rows = list(csv.DictReader(f))
            results[version].append([statistics.median(float(r[key]) for r in rows) for key, _ in COLUMNS])

    # The table: one line per run, then how much the new version changed each time.
    lines = ["", f"{scene_name}, {dots:,} dots",
             " " * 26 + "".join(f"{title:>24}" for _, title in COLUMNS)]
    for round_number in range(ROUNDS):
        for version in ["before", "after"]:
            times = results[version][round_number]
            lines.append(f"{names[version]:<17} run {round_number + 1}   " + "".join(f"{t:>21.3f} ms" for t in times))
    change = []
    for column in range(len(COLUMNS)):
        old = statistics.mean(run[column] for run in results["before"])
        new = statistics.mean(run[column] for run in results["after"])
        change.append((new - old) / old * 100.0)
    lines.append(f"{'change (minus = faster)':<26}" + "".join(f"{c:>+22.1f} %" for c in change))
    tables.append("\n".join(lines))

print("\n".join(tables))
